In [ ]:
from IPython.display import display

from config import SimConfig
from src.data_processing import load_and_cache_entire_fleet
from src.plants.hybrid_plant import FuelCellBatteryPlant
from src.utils.evaluation import VoyageBenchmarker
from src.plotting import plot_dashboard

In [ ]:
print("Initializing MARINER Validation Environment...")

# 1. Single unified configuration
config = SimConfig() 

# 2. Load the 1 Hz SOV dataset into RAM
fleet_cache = load_and_cache_entire_fleet(config)

# 3. Instantiate the benchmarking engine
benchmarker = VoyageBenchmarker(fleet_cache, exclude_days=[1, 2, 3])

# 4. The single physical truth: The Hybrid Plant
plant = FuelCellBatteryPlant(config)

In [ ]:
approaches = {
    # The Advanced Strategy (Actively manages battery SoC)
        "DP_Hybrid_Mean": {
            "strategy": "SDP",
            "sdp_variant": "MEAN_PROXY",
            "is_hybrid": True,   
            "config": config,
            "plant": plant
        },
    
    "DP_Hybrid_Tensor": {
        "strategy": "SDP",
        "sdp_variant": "TENSOR_SWEEP",
        "is_hybrid": True,   
        "config": config,
        "plant": plant
    },
    
    # The Wrapped Baseline DP (Blind to battery, optimizes H2 only)
    "DP_Baseline": {
        "strategy": "SDP",
        "is_hybrid": False,  # Triggers NaiveHybridWrapper 
        "config": config,
        "plant": plant
    },
    
    # The Wrapped Rule-Based Baseline (Moving threshold, blind to battery)
    "Heuristic_Baseline": {
        "strategy": "HEURISTIC",
        "is_hybrid": False,  # Triggers NaiveHybridWrapper 
        "config": config,
        "plant": plant
    }
}

In [ ]:
train_days=[4, 5, 6, 7, 8, 9, 10, 11, 12, 13] 
test_day=14

# Execute the simulations (All running inside the high-fidelity HybridSimulator)
df_results, sims = benchmarker.compare_approaches(approaches, train_days, test_day)

# Display tabular economics
print("\n--- Financial & Physical Summary ---")
display(df_results)

# Generate identical 4-panel dashboards for comparison
for approach_name, sim_instance in sims.items():
    plot_dashboard(
        sim=sim_instance, 
        approach_name=approach_name, 
        test_day=test_day, 
        layout='grid'
    )


In [ ]:
# Run Chronological Forward Chaining for the Hybrid Mean SDP
# print("--- APPROACH B: CHRONOLOGICAL FORWARD CHAINING ---")
# df_forward = benchmarker.run_forward_chaining(hybrid_mean_sdp, min_train_days=1)
# display(df_forward)

# Run Leave-One-Out for the Hybrid Mean SDP
print("\n--- APPROACH A: LEAVE ONE OUT ---")
df_loo_mean = benchmarker.run_leave_one_out(approaches["DP_Hybrid_Mean"])
display(df_loo_mean)

In [ ]:
# Run Leave-One-Out for the Hybrid Tensor SDP
print("\n--- APPROACH A: LEAVE ONE OUT ---")
df_loo_tensor = benchmarker.run_leave_one_out(approaches["DP_Hybrid_Tensor"])
display(df_loo_tensor)